# Data Preprocessing

- Raw abstract text (long)
- CLEAN the text
- remove junk characters
- CHUNK the text
- split into ~512 word pieces
- Ready for embeddings 

### Why do we chunk?
Embedding models have a token limit
You can't embed an entire paper at once
Smaller chunks = more precise retrieval
"Find the exact paragraph about tau protein"
vs
"Find the entire paper somewhere"

### Load Your Saved Papers

In [31]:
# Cell 1: Load the papers we fetched from PubMed
# We saved them in Block 1 as JSON

import json

# Load the papers
with open("../data/raw/alzheimer_papers.json", "r") as f:
    papers = json.load(f)

print(f"Loaded {len(papers)} papers")
print("\nFirst paper title:")
print(papers[0]["title"])
print("\nAbstract length (characters):")
print(len(papers[0]["abstract"]))

Loaded 100 papers

First paper title:
Bridging pathologies: Mechanistic insights into the diabetes-Alzheimer's nexus.

Abstract length (characters):
1344


### Clean The Text

You saved 100 papers in Block 1
Each paper has:
  - pmid      → unique ID
  - title     → paper title
  - abstract  → the long text
  - source    → "PubMed"

In Block 2 we're going to:
  - Take that abstract text
  - Clean it up
  - Split it into chunks
  - Each chunk ~512 words

Why 512?
  Because our embedding model
  "all-MiniLM-L6-v2" works best
  with text that size ✅

In [30]:
# Cell 2: Clean the abstract text
# Remove junk characters, extra spaces, encoding issues
#re.sub()     → find and replace using patterns
#\s+          → one or more whitespace characters
#\w\s\.\,     → keep letters, spaces, punctuation
#strip()      → remove spaces from start and end

import re

def clean_text(text):
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text)
    # Remove special characters but keep punctuation
    text = re.sub(r'[^\w\s\.\,\;\:\!\?\-\(\)]', ' ', text)
    # Strip leading/trailing spaces
    text = text.strip()
    return text

# Test it on first paper
sample = papers[0]["abstract"]
cleaned = clean_text(sample)



### Use pydantic for Automatic data validation
Manual approach: You must remember to call clean_text() separately on each abstract
Pydantic approach: The cleaning happens automatically during model instantiation via the @field_validator
Pydantic approach: Explicit validation errors with detailed messages about what's wrong

In [29]:
from pydantic import BaseModel, field_validator
import re

class Paper(BaseModel):
    pmid: str
    title: str
    abstract: str
    source: str

    @field_validator('abstract', mode='before')
    @classmethod
    def clean_abstract(cls, v):
        if not isinstance(v, str):
            return v
        # Remove extra whitespace
        v = re.sub(r'\s+', ' ', v)
        # Remove special characters but keep punctuation
        v = re.sub(r'[^\w\s\.\,\;\:\!\?\-\(\)]', ' ', v)
        # Strip leading/trailing spaces
        return v.strip()

# Test it
sample_paper = Paper(**papers[0])
print(f"Cleaned abstract: {sample_paper.abstract[:100]}")

Cleaned abstract: Type 2 diabetes mellitus (T2DM) is increasingly recognized as a major risk factor for Alzheimer s di


In [33]:
# Verification using clean_text function
print("CLEANING VERIFICATION:")
print("=" * 40)

# Use your actual function!
sample_paper = Paper(**papers[0])
print(f"Cleaned abstract: {sample_paper.abstract[:100]}")

# Check 1: No apostrophes
has_apostrophe = "'" in cleaned
print(f"Apostrophes removed: {not has_apostrophe} ✅")

# Check 2: Special chars remaining
import re
special_chars = re.findall(
    r'[^a-zA-Z0-9\s.,;:!?()\-]',
    cleaned
)
print(f"Special chars remaining: {special_chars[:5]}")

# Check 3: Length comparison
print(f"\nBefore length: {len(sample)}")
print(f"After length:  {len(cleaned)}")
print(f"Chars removed: {len(sample) - len(cleaned)}")

# Check 4: Sample comparison
print(f"\nBEFORE: {sample_paper.abstract[:100]}")
print(f"AFTER:  {sample_paper.abstract[:100]}")

CLEANING VERIFICATION:
Cleaned abstract: Type 2 diabetes mellitus (T2DM) is increasingly recognized as a major risk factor for Alzheimer s di
Apostrophes removed: True ✅
Special chars remaining: ['β']

Before length: 1344
After length:  1344
Chars removed: 0

BEFORE: Type 2 diabetes mellitus (T2DM) is increasingly recognized as a major risk factor for Alzheimer s di
AFTER:  Type 2 diabetes mellitus (T2DM) is increasingly recognized as a major risk factor for Alzheimer s di


### Chunking

In [34]:
# Cell 3: Split text into chunks
# Why? Embedding models work best with ~200 word pieces

def chunk_text(text, chunk_size=200, overlap=50):
    
    words = text.split()
    chunks = []
    start = 0
    
    # ← ADD THIS GUARD!
    if len(words) < 30:
        return [text]
    
    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start = end - overlap
    
    return chunks

# Test on first paper
# Load papers if not defined
import json
with open("../data/raw/alzheimer_papers.json", "r") as f:
    papers = json.load(f)

sample_abstract = papers[0]["abstract"]
chunks = chunk_text(sample_abstract)

print(f"Abstract word count: {len(sample_abstract.split())}")
print(f"Number of chunks: {len(chunks)}")
print(f"\nChunk 1:\n{chunks[0]}")

Abstract word count: 165
Number of chunks: 2

Chunk 1:
Type 2 diabetes mellitus (T2DM) is increasingly recognized as a major risk factor for Alzheimer's disease (AD), with mounting evidence highlighting shared pathophysiological mechanisms. This review explores the intricate biological and molecular links between these two chronic disorders. Key overlapping pathways include impaired insulin signaling, chronic inflammation, oxidative stress, mitochondrial dysfunction, amyloid-beta (Aβ) accumulation, tau hyperphosphorylation, and the formation of advanced glycation end-products (AGEs). Disruption of insulin signaling in the brain contributes to synaptic loss and neurodegeneration, while systemic metabolic disturbances aggravate blood-brain barrier dysfunction and neurovascular damage. Emerging studies also underscore the role of antidiabetic treatments, especially newer agents targeting the gut-brain axis, in modulating AD progression. The review further examines preclinical models, clin

### Process ALL Papers

In [37]:
# Cell 4: Chunk all 100 papers and structure the data
# This creates our final dataset ready for embeddings

all_chunks = []

for paper in papers:
    
    # Combine title + abstract for richer context
    full_text = paper["title"] + ". " + paper["abstract"]
    
    # Clean it
    # Cell 4: Chunk all 100 papers and structure the data
    # This creates our final dataset ready for embeddings

    all_chunks = []

    for paper in papers:
        
        # Combine title + abstract for richer context
        full_text = paper["title"] + ". " + paper["abstract"]
        
        # Clean it
        cleaned = clean_text(full_text)
        
        # Chunk it
        chunks = chunk_text(cleaned)
        
        # Store each chunk with its metadata
        for i, chunk in enumerate(chunks):
            all_chunks.append({
                "pmid": paper["pmid"],
                "chunk_id": i,
                "text": chunk,
                "source": "PubMed",
                "title": paper["title"]
            })

    print(f"Total papers: {len(papers)}")
    print(f"Total chunks: {len(all_chunks)}")
    print(f"\nSample chunk:")
    print(all_chunks[0])
    
    # Chunk it
    chunks = chunk_text(cleaned)
    
    # Store each chunk with its metadata
    for i, chunk in enumerate(chunks):
        all_chunks.append({
            "pmid": paper["pmid"],
            "chunk_id": i,
            "text": chunk,
            "source": "PubMed",
            "title": paper["title"]
        })

print(f"Total papers: {len(papers)}")
print(f"Total chunks: {len(all_chunks)}")
print(f"\nSample chunk:")
print(all_chunks[0])

Total papers: 100
Total chunks: 160

Sample chunk:
{'pmid': '41834915', 'chunk_id': 0, 'text': 'Bridging pathologies: Mechanistic insights into the diabetes-Alzheimer s nexus.. Type 2 diabetes mellitus (T2DM) is increasingly recognized as a major risk factor for Alzheimer s disease (AD), with mounting evidence highlighting shared pathophysiological mechanisms. This review explores the intricate biological and molecular links between these two chronic disorders. Key overlapping pathways include impaired insulin signaling, chronic inflammation, oxidative stress, mitochondrial dysfunction, amyloid-beta (Aβ) accumulation, tau hyperphosphorylation, and the formation of advanced glycation end-products (AGEs). Disruption of insulin signaling in the brain contributes to synaptic loss and neurodegeneration, while systemic metabolic disturbances aggravate blood-brain barrier dysfunction and neurovascular damage. Emerging studies also underscore the role of antidiabetic treatments, especially new

In [38]:
# Check all abstract lengths
lengths = [
    len(p["abstract"].split()) 
    for p in papers
]

print(f"Average words: {sum(lengths)//len(lengths)}")
print(f"Shortest: {min(lengths)} words")
print(f"Longest:  {max(lengths)} words")
print(f"Under 50 words:  {sum(1 for l in lengths if l < 50)}")
print(f"50-150 words:    {sum(1 for l in lengths if 50 <= l < 150)}")
print(f"Over 150 words:  {sum(1 for l in lengths if l >= 150)}")

# Show a LONG abstract example
long_paper = max(papers, key=lambda p: len(p["abstract"].split()))
print(f"\nLongest abstract ({len(long_paper['abstract'].split())} words):")
print(long_paper["abstract"][:300])


Average words: 141
Shortest: 3 words
Longest:  503 words
Under 50 words:  23
50-150 words:    28
Over 150 words:  49

Longest abstract (503 words):
The blood-brain barrier (BBB) is a major obstacle to the development of brain-targeted drug delivery systems, restricting greater than 98% of small molecules (<500 Da) and virtually all large-molecule drugs from entering the brain tissues from the bloodstream, resulting in suboptimal drug doses and 


### Save the Chunks

In [40]:
# Cell 5: Save chunks to file
# This is the input for Block 3 (Embeddings)

import json
import os

os.makedirs("../data/processed", exist_ok=True)

with open("../data/processed/chunks.json", "w") as f:
    json.dump(all_chunks, f, indent=2)

print(f"Saved {len(all_chunks)} chunks")
print("\nSample chunk structure:")
if all_chunks:
    print(json.dumps(all_chunks[0], indent=2))
else:
    print("No chunks to display.")

Saved 161 chunks

Sample chunk structure:
{
  "pmid": "41834915",
  "chunk_id": 0,
  "text": "Bridging pathologies: Mechanistic insights into the diabetes-Alzheimer s nexus.. Type 2 diabetes mellitus (T2DM) is increasingly recognized as a major risk factor for Alzheimer s disease (AD), with mounting evidence highlighting shared pathophysiological mechanisms. This review explores the intricate biological and molecular links between these two chronic disorders. Key overlapping pathways include impaired insulin signaling, chronic inflammation, oxidative stress, mitochondrial dysfunction, amyloid-beta (A\u03b2) accumulation, tau hyperphosphorylation, and the formation of advanced glycation end-products (AGEs). Disruption of insulin signaling in the brain contributes to synaptic loss and neurodegeneration, while systemic metabolic disturbances aggravate blood-brain barrier dysfunction and neurovascular damage. Emerging studies also underscore the role of antidiabetic treatments, especially 

### Check Your Saved File:

**BLOCK 2 COMPLETE!**

- ✅ Loaded papers from JSON
- ✅ Cleaned the text
- ✅ Chunked into 512 word pieces
- ✅ Combined title + abstract
- ✅ Saved to processed folder